# Reverse JAKET: 32-dimensional Colab pilot
Select **Runtime -> Change runtime type -> GPU**. Run cells individually.
This notebook uses FB15k-237, entity and relation soft prompts, frozen RoBERTa,
direct projection into the second RGAT encoder, ComplEx and plain BCE.

Start with a ZIP of the updated local source. GitHub cloning works only after
these changes have been pushed. Exclude `.git`, cached models/data and old outputs
from the ZIP. The initial pilot is a software/resource check, not research evidence.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

SOURCE = "upload"  # "upload" (updated ZIP) or "github" (after pushing changes)
REPO_URL = "https://github.com/pubudu2003060/research_project.git"
REF = ""  # optional published branch or commit for the github option

if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one project ZIP")
    destination = Path("/content/reverse_jaket_source").resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError("ZIP contains an unsafe path")
        archive.extractall(destination)
    candidates = list(destination.rglob("run_experiment.py"))
    if len(candidates) != 1:
        raise ValueError("Expected exactly one project containing run_experiment.py")
    project = candidates[0].parent
elif SOURCE == "github":
    project = Path("/content/research_project")
    if not project.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project)])
    if REF:
        subprocess.check_call(["git", "checkout", REF], cwd=project)
else:
    raise ValueError("SOURCE must be upload or github")
os.chdir(project)
assert Path("run_colab_preflight.py").exists(), "This is the old source; upload the updated project"
print("Project:", project)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import torch
import yaml
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print(torch.__version__, torch.cuda.get_device_name(0))
from training.config import load_config
assert load_config("experiments/configs/proposed_complex.yaml").kg_dim == 32
subprocess.check_call([sys.executable, "run_pipeline_smoke_test.py"])


## Persistent outputs and pilot configuration
Drive holds checkpoints and configs. Downloaded data and LM caches stay on the
local runtime. Repeat setup after a runtime reset. Use a new `RUN_NAME` for a new
experiment; use resume for an existing experiment.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "pilot32"
ROOT = Path("/content/drive/MyDrive/reverse_jaket")
ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = ROOT / f"{RUN_NAME}.yaml"
RUN = ROOT / RUN_NAME
if CONFIG.exists():
    cfg = yaml.safe_load(CONFIG.read_text())
    print("Keeping saved configuration for reproducibility:", CONFIG)
else:
    cfg = yaml.safe_load(Path("experiments/configs/proposed_complex.yaml").read_text())
    cfg.update(warmup_epochs=1, max_epochs=3, eval_every=1,
               early_stopping_patience=0, output_dir=str(RUN))
    CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))


## Real-data preflight
This downloads real triples, aligned entity/relation text and pretrained
RoBERTa, then runs one semantic forward/backward/optimizer step. It reports
step time and peak CUDA allocation. The largest training batch and full
validation can use more memory. Reduce LM micro-batch first if needed and start
a new experiment when changing settings. No full-GPU feasibility claim is made
until you measure it here.


In [ ]:
subprocess.check_call([sys.executable, "run_colab_preflight.py", str(CONFIG)])


## Optional AMP diagnostics and performance profiles
The updated trainer logs the first batch and every 50 batches. It encodes only
relations required by sampled edges or scored triples; every required prompt
still depends on the current KG vectors. It caches token IDs/masks only.

For the reported FP16 failure, enable the diagnostic cell below. It records
missing/zero/NaN/Inf gradients and loss-scaler backoff. The first attempt with
legacy computation and no retries may fail intentionally; the cell continues
to the checked optimized path. No CUDA stability claim is made from CPU tests.

Profiles are separate disposable training runs with identical initial weights,
sampled batches and seeds. They do not alter the pilot's artifacts. Defaults
keep profiling disabled and use only five batches when enabled. Full-epoch
profiling runs two entire semantic epochs plus validation and can take hours.
See `PERFORMANCE_NOTES.md` for interpretation and limitations.


In [ ]:
RUN_AMP_DIAGNOSTICS = False
if RUN_AMP_DIAGNOSTICS:
    import time
    DIAG = ROOT / "diagnostics"
    DIAG.mkdir(parents=True, exist_ok=True)
    stamp = str(time.time_ns())
    cases = [
        ("legacy_fp16_first_attempt", ["--legacy", "--amp-dtype", "float16", "--amp-max-retries", "0"]),
        ("optimized_fp16", ["--amp-dtype", "float16"]),
    ]
    if torch.cuda.is_bf16_supported():
        cases.append(("optimized_bf16", ["--amp-dtype", "bfloat16"]))
    for name, options in cases:
        output = DIAG / f"{name}_{stamp}.json"
        completed = subprocess.run([sys.executable, "run_colab_preflight.py", str(CONFIG),
                                    *options, "--output", str(output)])
        print(name, "exit code:", completed.returncode, "report:", output)


In [ ]:
RUN_PROFILES = False
PROFILE_FULL_EPOCH = False
PROFILE_PRECISION = None  # None keeps saved config; or "float32", "float16", "bfloat16"
if RUN_PROFILES:
    import time
    import json
    DIAG = ROOT / "diagnostics"
    DIAG.mkdir(parents=True, exist_ok=True)
    PROFILE_OUTPUT = DIAG / f"semantic_profile_{time.time_ns()}.json"
    cases = (["legacy", "optimized"] if PROFILE_FULL_EPOCH else
             ["legacy", "optimized", "no_checkpoint", "larger_relation_batch"])
    command = [sys.executable, "profile_semantic.py", str(CONFIG),
               "--output", str(PROFILE_OUTPUT), "--cases", *cases]
    command += ["--full-epoch"] if PROFILE_FULL_EPOCH else ["--batches", "5"]
    if PROFILE_PRECISION:
        command += ["--amp-dtype", PROFILE_PRECISION]
    warmup = RUN / "warmup_checkpoint.pt"
    if warmup.exists():
        command += ["--checkpoint", str(warmup)]
    completed = subprocess.run(command)
    print("Profiler exit code:", completed.returncode, "report:", PROFILE_OUTPUT)
    if PROFILE_OUTPUT.exists():
        import pandas as pd
        report = json.loads(PROFILE_OUTPUT.read_text())
        display(pd.DataFrame([
            {"case": name, "status": r["status"],
             "first_batch_s": r.get("cold_batch_seconds"),
             "steady_batch_s": r.get("steady_batch_mean_seconds"),
             "estimated_training_epoch_s": r.get("estimated_training_epoch_seconds"),
             "measured_epoch_s": r.get("measured_epoch_seconds"),
             "peak_GPU_MiB": r.get("training_memory", {}).get("cuda_peak_mib")}
            for name, r in report["cases"].items()
        ]))


## Train or resume the pilot
This may take substantial time even at width 32: RoBERTa processes sampled
entities and relations every batch. The first epoch is structural warm-up;
subsequent epochs use the complete semantic model. Re-running this cell resumes
the last epoch checkpoint, or preserves an already completed run.


In [ ]:
import json
status_path = RUN / "run_status.json"
completed = status_path.exists() and json.loads(status_path.read_text()).get("status") == "complete"
if completed:
    print("Already complete:", RUN / "final_results.json")
else:
    command = [sys.executable, "run_experiment.py", str(CONFIG), "--output-dir", str(RUN)]
    checkpoint = RUN / "last_checkpoint.pt"
    if checkpoint.exists():
        command += ["--resume", str(checkpoint)]
    subprocess.check_call(command)


In [ ]:
subprocess.check_call([sys.executable, "evaluate_checkpoint.py", str(RUN / "best_checkpoint.pt")])
subprocess.check_call([sys.executable, "summarize_results.py", str(RUN),
                       "--output-dir", str(ROOT / f"{RUN_NAME}_report")])
print(json.dumps(json.loads((RUN / "final_results.json").read_text()), indent=2))


## Research runs after the pilot
The following cell prepares **four matched 32-dimensional experiments** with
seed 42 using the normal training budget (up to 300 epochs; 10 warm-up epochs).
Set `EXECUTE = True` only when ready to run them. This includes KG-only,
text-enhanced, proposed, and no-warm-up variants. The full default comparison
plan adds seeds 43/44 and dimensions 128/200/256; see README before scaling up.
Keep the pilot separate from those comparisons. Old gated-fusion checkpoints
cannot resume this architecture. Record text quality/coverage, GPU memory,
runtime, final metrics and uncertainty across seeds.


In [ ]:
EXECUTE = False
COMPARISONS = ROOT / "comparisons32"
command = [sys.executable, "run_comparisons.py", "--output-root", str(COMPARISONS),
           "--seeds", "42", "--dimensions"]
if EXECUTE:
    command.append("--execute")
subprocess.check_call(command)
if EXECUTE:
    subprocess.check_call([sys.executable, "summarize_results.py", str(COMPARISONS),
                           "--output-dir", str(ROOT / "comparisons32_report")])


The implementation follows the PDF methodology with project-approved
FB15k-237, RGAT, ComplEx-only, and initial width 32. The PDF does not specify the
exact relation-update formula; README documents the chosen graph readout and
sampled training protocol. Full research experiments and report reconciliation
remain. Colab resources and session lengths vary: [official FAQ](https://research.google.com/colaboratory/faq.html).
